In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from flask import Flask, request, render_template_string
import threading
import urllib.request
import logging
import json
import math
import time

# ==============================================================================
# PHẦN 1: CORE MACHINE LEARNING ENGINE - KAFORECAST ENGINE
# Hệ thống lõi xử lý dữ liệu và AI dự báo doanh thu, tính toán % ảnh hưởng.
# Được thiết kế theo chuẩn hướng đối tượng (OOP) với các lớp bảo vệ dữ liệu.
# ==============================================================================
class CoffeeBusinessIntelligence:
    """
    KAFORECAST CORE ENGINE:
    Mô-đun AI học máy chịu trách nhiệm phân tích hồi quy tuyến tính (Linear Regression).
    Tính toán dự báo doanh thu dựa trên các biến phân loại và biến liên tục.
    """
    def __init__(self):
        self._model = LinearRegression()
        self._is_trained = False
        self._feature_names = [
            'gia_trung_binh', 'luong_khach', 'vi_tri',
            'cho_ngoi', 'delivery', 'nhan_vien',
            'sinh_vien_ngoi_lau', 'dien_tich', 'doi_thu_500m'
        ]
        self._coefficients = []
        self._intercept = 0.0

    @property
    def is_trained(self):
        """Kiểm tra trạng thái huấn luyện của mô hình KAFORECAST."""
        return self._is_trained

    def create_synthetic_dataset(self, samples=120):
        np.random.seed(42)
        print(f"[KAFORECAST] Đang tiến hành giả lập {samples} mẫu dữ liệu không gian kinh doanh...")

        # 1. Khởi tạo các vector dữ liệu độc lập
        gia_trung_binh = np.random.uniform(20000, 80000, samples)
        luong_khach = np.random.randint(20, 500, samples)
        vi_tri = np.random.choice([0, 1, 2], samples)
        cho_ngoi = np.random.choice([0, 1], samples)
        delivery = np.random.choice([0, 1], samples)
        nhan_vien = np.random.randint(1, 15, samples)
        sinh_vien_ngoi_lau = np.random.choice([0, 1], samples)
        dien_tich = np.random.uniform(10, 250, samples)
        doi_thu_500m = np.random.randint(0, 20, samples)

        # 2. Xây dựng Dataframe
        data = {
            'gia_trung_binh': gia_trung_binh,
            'luong_khach': luong_khach,
            'vi_tri': vi_tri,
            'cho_ngoi': cho_ngoi,
            'delivery': delivery,
            'nhan_vien': nhan_vien,
            'sinh_vien_ngoi_lau': sinh_vien_ngoi_lau,
            'dien_tich': dien_tich,
            'doi_thu_500m': doi_thu_500m
        }
        df = pd.DataFrame(data)

        # 3. Kỹ thuật đặc trưng (Feature Engineering) cho Biến Mục Tiêu
        base_revenue = df['luong_khach'] * df['gia_trung_binh'] * 0.82

        location_bonus = df['vi_tri'].map({0: 0, 1: 800000, 2: 1500000})
        seating_bonus = df['cho_ngoi'] * 1200000
        delivery_bonus = df['delivery'] * 900000
        space_bonus = df['dien_tich'] * 15000

        student_penalty = df['sinh_vien_ngoi_lau'] * -500000
        competition_penalty = df['doi_thu_500m'] * -400000

        market_noise = np.random.normal(0, 1000000, samples)

        df['doanh_thu_ngay'] = (
            base_revenue + location_bonus + seating_bonus +
            delivery_bonus + space_bonus + student_penalty +
            competition_penalty + market_noise
        )

        df['doanh_thu_ngay'] = df['doanh_thu_ngay'].apply(lambda x: max(0, x))
        return df

    def train_engine(self):
        """Huấn luyện KAFORECAST Engine bằng thuật toán hồi quy."""
        start_time = time.time()
        dataset = self.create_synthetic_dataset(samples=1200)

        X = dataset[self._feature_names]
        y = dataset['doanh_thu_ngay']

        print("[KAFORECAST] Bắt đầu quá trình khớp dữ liệu (Fitting Model)...")
        self._model.fit(X, y)
        self._is_trained = True

        self._coefficients = self._model.coef_.tolist()
        self._intercept = float(self._model.intercept_)

        elapsed_time = round(time.time() - start_time, 2)
        print(f"--- [KAFORECAST] Huấn luyện hoàn tất trong {elapsed_time}s ---")
        print(f"--- [KAFORECAST] Hệ số giao điểm (Intercept): {self._intercept} ---")

    def run_inference(self, inputs):
        """Dự báo doanh thu và tính toán phân bổ phần trăm ảnh hưởng từ các đầu vào."""
        if not self.is_trained:
            raise RuntimeError("LỖI KAFORECAST: Mô hình chưa được khởi tạo và huấn luyện!")

        input_array = np.array(inputs).reshape(1, -1)
        raw_pred = self._model.predict(input_array)[0]
        final_revenue = max(0, float(raw_pred))

        impact_raw = [abs(inputs[i] * self._coefficients[i]) for i in range(len(inputs))]
        total_impact = sum(impact_raw)

        if total_impact > 0:
            impact_percentages = [round((val / total_impact) * 100, 2) for val in impact_raw]
        else:
            impact_percentages = [0.0] * len(inputs)

        return final_revenue, impact_percentages

kaforecast_engine = CoffeeBusinessIntelligence()
kaforecast_engine.train_engine()

# ==============================================================================
# PHẦN 2: FLASK WEB APP & KAFORECAST HTML GIAO DIỆN
# ==============================================================================
app = Flask(__name__)

HTML_DASHBOARD = """
<!DOCTYPE html>
<html lang="vi">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>KAFORECAST | Real-Time Spatial F&B</title>

    <!-- CSS Tích hợp -->
    <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/css/bootstrap.min.css" rel="stylesheet">
    <link href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.0.0/css/all.min.css" rel="stylesheet">
    <link href="https://fonts.googleapis.com/css2?family=Be+Vietnam+Pro:wght@300;400;500;600;800;900&display=swap" rel="stylesheet">

    <!-- Thư viện Bản đồ (Leaflet) -->
    <link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css" crossorigin=""/>

    <!-- Thư viện Biểu đồ (Chart.js) -->
    <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>

    <style>
        :root {
            /* SỬA MÀU: Đổi sang Theme Sáng (Light Theme) */
            --bg-grad-1: #f1f5f9; --bg-grad-2: #e2e8f0; --bg-grad-3: #cbd5e1;
            --glass-bg: rgba(255, 255, 255, 0.85); --glass-border: rgba(255, 255, 255, 1);
            --neon-accent: #0284c7; --neon-primary: #0ea5e9; --neon-alert: #ef4444;
            --text-main: #0f172a; --text-muted: #475569;
        }

        body {
            font-family: 'Be Vietnam Pro', sans-serif;
            background: linear-gradient(135deg, var(--bg-grad-1), var(--bg-grad-2), var(--bg-grad-3));
            background-attachment: fixed; color: var(--text-main); min-height: 100vh;
            padding: 40px 0; overflow-x: hidden;
        }

        .glass-panel {
            background: var(--glass-bg); backdrop-filter: blur(25px); border: 1px solid var(--glass-border);
            border-radius: 24px; padding: 40px; box-shadow: 0 25px 50px rgba(0, 0, 0, 0.1);
            margin-bottom: 30px; animation: fadeIn 0.8s ease-out forwards;
        }

        @keyframes fadeIn { from { opacity: 0; transform: translateY(20px); } to { opacity: 1; transform: translateY(0); } }

        .brand-logo {
            font-weight: 900; font-size: 3rem; background: linear-gradient(120deg, #0284c7 0%, #0ea5e9 100%);
            -webkit-background-clip: text; -webkit-text-fill-color: transparent; text-transform: uppercase;
        }

        /* SỬA MÀU: Chỉnh Input hiển thị tốt trên nền sáng */
        .form-floating > .form-control, .form-floating > .form-select {
            background: #ffffff; border: 1px solid #cbd5e1; color: var(--text-main);
            font-weight: 600; border-radius: 12px; transition: all 0.3s; height: calc(3.5rem + 5px);
        }

        .form-floating > .form-control:focus, .form-floating > .form-select:focus {
            background: #f8fafc; border-color: var(--neon-accent);
            box-shadow: 0 0 18px rgba(2, 132, 199, 0.2); color: var(--text-main);
        }

        .form-floating label { color: var(--text-muted); font-weight: 500; padding-left: 20px; }
        .input-icon { position: absolute; top: 50%; right: 15px; transform: translateY(-50%); color: var(--neon-accent); pointer-events: none; }

        .btn-glow {
            background: linear-gradient(45deg, #0284c7, #0ea5e9); border: none; color: white;
            padding: 18px 45px; font-size: 1.2rem; font-weight: 800; border-radius: 50px;
            transition: all 0.4s ease; box-shadow: 0 10px 30px rgba(2, 132, 199, 0.3);
            text-transform: uppercase; letter-spacing: 2px;
        }
        .btn-glow:hover { transform: translateY(-4px) scale(1.02); box-shadow: 0 20px 40px rgba(2, 132, 199, 0.5); color: #fff; }

        #map-container {
            width: 100%; height: 480px; border-radius: 16px; border: 2px solid var(--glass-border);
            margin-bottom: 20px; overflow: hidden; position: relative; background: #e2e8f0;
        }
        #map { width: 100%; height: 100%; z-index: 1; }

        .spatial-info {
            background: rgba(2, 132, 199, 0.1); border-left: 5px solid var(--neon-accent);
            padding: 18px 25px; border-radius: 0 12px 12px 0; margin-top: 15px; display: none;
        }

        .money-text { font-size: 3.8rem; font-weight: 900; color: #0284c7; text-shadow: 0 0 25px rgba(2, 132, 199, 0.2); }
        .chart-container { position: relative; height: 380px; width: 100%; }

        #map-loading {
            position: absolute; top: 0; left: 0; width: 100%; height: 100%;
            background: rgba(255, 255, 255, 0.85); z-index: 1000;
            display: flex; flex-direction: column; justify-content: center; align-items: center;
            color: var(--neon-accent); display: none; backdrop-filter: blur(5px);
        }

        /* Ghi đè text Bootstrap cho nền sáng */
        .text-white { color: var(--text-main) !important; }
        .bg-dark { background-color: #ffffff !important; border-color: #cbd5e1 !important; }
        .text-info { color: #0284c7 !important; }
        .border-secondary { border-color: #cbd5e1 !important; }
    </style>
</head>
<body>

<div class="container">
    <div class="row justify-content-center">
        <div class="col-xl-11">
            <div class="glass-panel">
                <div class="text-center mb-5">
                    <h1 class="brand-logo">KAFORECAST</h1>
                    <p class="text-muted mt-2" style="font-size: 1.15rem; font-weight: 500;">
                        Hệ thống Trí tuệ Không gian Thực (Real-Time Spatial BI) cho ngành F&B
                    </p>
                </div>

                {% if error %}
                <div class="alert alert-danger">
                    <i class="fa-solid fa-triangle-exclamation fa-lg me-2"></i> <strong>LỖI HỆ THỐNG:</strong> {{ error }}
                </div>
                {% endif %}

                <form method="POST" action="/process" id="aiForm">
                    <!-- INPUT ẨN BẢO TOÀN TRẠNG THÁI BẢN ĐỒ -->
                    <input type="hidden" id="lat_input" name="lat" value="{{ request.form.get('lat', '10.7769') }}">
                    <input type="hidden" id="lng_input" name="lng" value="{{ request.form.get('lng', '106.7009') }}">

                    <!-- MODULE 1: QUÉT VỊ TRÍ THỰC TẾ (OVERPASS API) -->
                    <h4 class="mb-4" style="color: var(--neon-accent); border-bottom: 2px solid rgba(0,0,0,0.1); padding-bottom: 12px; font-weight: 700;">
                        <i class="fa-solid fa-satellite fa-bounce me-2"></i> Bước 1: Quét Dữ Liệu Vệ Tinh Trực Tiếp
                    </h4>

                    <div class="row mb-5">
                        <div class="col-12">
                            <div class="input-group mb-3 shadow-sm">
                                <span class="input-group-text bg-dark text-white border-secondary"><i class="fa-solid fa-magnifying-glass"></i></span>
                                <input type="text" id="addressSearch" class="form-control bg-dark text-white border-secondary" placeholder="Nhập địa chỉ thật để bay tới (VD: 1 Lê Duẩn, Quận 1)..." style="font-size: 1.1rem;">
                                <button class="btn btn-outline-info text-white" type="button" id="btnSearchLocation" style="font-weight: 700; padding: 0 25px; border: 1px solid #cbd5e1;">TÌM TỌA ĐỘ</button>
                            </div>

                            <div id="map-container">
                                <div id="map"></div>
                                <div id="map-loading">
                                    <div class="spinner-border text-info" role="status" style="width: 3rem; height: 3rem;"></div>
                                    <div class="mt-3 fw-bold" id="loadingText" style="color: var(--text-main); font-size: 1.2rem;">Đang kết nối vệ tinh Overpass API...</div>
                                </div>
                            </div>

                            <div class="spatial-info" id="spatialInfoBox">
                                <h5 class="fw-bold text-white mb-2" id="spatialTitle"><i class="fa-solid fa-microchip text-info"></i> Kết quả phân tích Thực Tế:</h5>
                                <p class="mb-0 text-white" id="spatialDetails">Đang chờ nhận tọa độ...</p>
                            </div>
                        </div>
                    </div>

                    <!-- MODULE 2: THÔNG SỐ VẬN HÀNH KINH DOANH -->
                    <h4 class="mt-5 mb-4" style="color: var(--neon-accent); border-bottom: 2px solid rgba(0,0,0,0.1); padding-bottom: 12px; font-weight: 700;">
                        <i class="fa-solid fa-sliders me-2"></i> Bước 2: Thiết Lập Tham Số Vận Hành
                    </h4>

                    <div class="row g-4">
                        <div class="col-md-4 position-relative">
                            <div class="form-floating">
                                <input type="number" step="1000" class="form-control" id="f1" name="f1" placeholder="Giá bán" value="{{ request.form.get('f1', '') }}" required>
                                <label for="f1">Giá trung bình/ly (VNĐ)</label>
                            </div>
                            <i class="fa-solid fa-tags input-icon"></i>
                        </div>

                        <div class="col-md-4 position-relative">
                            <div class="form-floating">
                                <input type="number" class="form-control" id="f2" name="f2" placeholder="Khách" value="{{ request.form.get('f2', '') }}" required>
                                <label for="f2">Lượng khách kỳ vọng/ngày</label>
                            </div>
                            <i class="fa-solid fa-users input-icon"></i>
                        </div>

                        <div class="col-md-4 position-relative">
                            <div class="form-floating">
                                <input type="number" class="form-control" id="f8" name="f8" placeholder="Diện tích" value="{{ request.form.get('f8', '') }}" required>
                                <label for="f8">Diện tích mặt bằng (m²)</label>
                            </div>
                            <i class="fa-solid fa-expand input-icon"></i>
                        </div>

                        <div class="col-md-4 position-relative">
                            <div class="form-floating">
                                <select class="form-select fw-bold" id="f3" name="f3" required style="border-left: 4px solid var(--neon-accent);">
                                    <option value="" disabled {% if not request.form.get('f3') %}selected{% endif %}>-- KAFORECAST tự động điền --</option>
                                    <option value="1" {% if request.form.get('f3') == '1' %}selected{% endif %}>Khu vực có Trường học / Làng Đại học</option>
                                    <option value="2" {% if request.form.get('f3') == '2' %}selected{% endif %}>Khu vực nhiều Văn phòng / Công sở</option>
                                    <option value="0" {% if request.form.get('f3') == '0' %}selected{% endif %}>Khu Dân cư thường / Không rõ ràng</option>
                                </select>
                                <label for="f3">Phân loại Vị trí (Dữ liệu Thực)</label>
                            </div>
                        </div>

                        <div class="col-md-4 position-relative">
                            <div class="form-floating">
                                <input type="number" class="form-control" id="f6" name="f6" placeholder="Nhân viên" value="{{ request.form.get('f6', '') }}" required>
                                <label for="f6">Số nhân sự làm việc/ca</label>
                            </div>
                            <i class="fa-solid fa-user-tie input-icon"></i>
                        </div>

                        <div class="col-md-4 position-relative">
                            <div class="form-floating">
                                <input type="number" class="form-control" id="f9" name="f9" placeholder="Đối thủ" value="{{ request.form.get('f9', '') }}" required>
                                <label for="f9">Số đối thủ (Bán kính 500m)</label>
                            </div>
                            <i class="fa-solid fa-store-slash input-icon"></i>
                        </div>

                        <div class="col-md-4 position-relative">
                            <div class="form-floating">
                                <select class="form-select" id="f4" name="f4" required>
                                    <option value="" disabled {% if not request.form.get('f4') %}selected{% endif %}></option>
                                    <option value="1" {% if request.form.get('f4') == '1' %}selected{% endif %}>Quán rộng, đầu tư nhiều chỗ ngồi</option>
                                    <option value="0" {% if request.form.get('f4') == '0' %}selected{% endif %}>Mô hình Kiosk / Take-away nhỏ</option>
                                </select>
                                <label for="f4">Mô hình Không gian</label>
                            </div>
                        </div>

                        <div class="col-md-4 position-relative">
                            <div class="form-floating">
                                <select class="form-select" id="f5" name="f5" required>
                                    <option value="" disabled {% if not request.form.get('f5') %}selected{% endif %}></option>
                                    <option value="1" {% if request.form.get('f5') == '1' %}selected{% endif %}>Có bán trên App (ShopeeFood, Grab...)</option>
                                    <option value="0" {% if request.form.get('f5') == '0' %}selected{% endif %}>Chỉ bán Offline tại quán</option>
                                </select>
                                <label for="f5">Chiến lược Giao hàng</label>
                            </div>
                        </div>

                        <div class="col-md-4 position-relative">
                            <div class="form-floating">
                                <select class="form-select" id="f7" name="f7" required>
                                    <option value="" disabled {% if not request.form.get('f7') %}selected{% endif %}></option>
                                    <option value="1" {% if request.form.get('f7') == '1' %}selected{% endif %}>Cho phép / Cung cấp ổ cắm, wifi mạnh</option>
                                    <option value="0" {% if request.form.get('f7') == '0' %}selected{% endif %}>Thiết kế hạn chế ngồi lâu</option>
                                </select>
                                <label for="f7">Chính sách Lưu trú</label>
                            </div>
                        </div>
                    </div>

                    <!-- NÚT CHỨC NĂNG -->
                    <div class="mt-5 d-flex justify-content-center gap-3 flex-wrap">
                        <a href="/" class="btn btn-outline-danger d-flex align-items-center px-4" style="border-radius: 50px; font-weight: 700;">
                            <i class="fa-solid fa-rotate-right me-2"></i> LÀM MỚI TỪ ĐẦU
                        </a>
                        <button type="submit" class="btn btn-glow" id="submitBtn">
                            <i class="fa-solid fa-bolt me-2"></i> KAFORECAST PREDICT
                        </button>
                    </div>
                </form>

                <!-- MODULE 3: KHU VỰC HIỂN THỊ KẾT QUẢ AI -->
                {% if prediction is not none %}
                <div class="glass-panel mt-5" style="border-color: rgba(2, 132, 199, 0.4); background: rgba(255, 255, 255, 0.95);" id="resultSection">
                    <div class="row align-items-center">
                        <div class="col-lg-6 text-center text-lg-start border-lg-end border-secondary pe-lg-4">
                            <span class="badge bg-success text-white px-3 py-2 rounded-pill mb-3" style="font-weight: 800; font-size: 0.9rem; letter-spacing: 1px;">
                                <i class="fa-solid fa-check-circle me-1"></i> BÁO CÁO THÀNH CÔNG
                            </span>
                            <h4 class="text-uppercase fw-bold" style="color: var(--text-muted);">Doanh Thu Đề Xuất</h4>
                            <div class="money-text">{{ "{:,.0f}".format(prediction).replace(',', '.') }} ₫</div>
                            <p class="text-muted mb-0" style="font-size: 1.1rem;">~ Doanh thu ước tính trung bình mỗi ngày</p>
                            <hr class="border-secondary my-4">
                            <div class="row text-center">
                                <div class="col-6">
                                    <p class="text-muted small mb-1 fw-bold">DỰ PHÓNG 30 NGÀY</p>
                                    <h5 class="fw-bold text-white fs-4">{{ "{:,.0f}".format(prediction * 30).replace(',', '.') }} ₫</h5>
                                </div>
                                <div class="col-6">
                                    <p class="text-muted small mb-1 fw-bold">DỰ PHÓNG 1 NĂM</p>
                                    <h5 class="fw-bold text-white fs-4">{{ "{:,.0f}".format(prediction * 365).replace(',', '.') }} ₫</h5>
                                </div>
                            </div>
                        </div>

                        <div class="col-lg-6 mt-4 mt-lg-0">
                            <h5 class="text-center mb-3 fw-bold" style="color: var(--neon-accent);">
                                <i class="fa-solid fa-radar me-2"></i> Trọng Số Ảnh Hưởng
                            </h5>
                            <div class="chart-container">
                                <canvas id="impactChart"></canvas>
                            </div>
                        </div>
                    </div>
                </div>
                {% endif %}
            </div>

            <div class="text-center mt-4 mb-5" style="font-size: 0.95rem; color: var(--text-muted); font-weight: 500;">
                <i class="fa-brands fa-python text-info"></i> Bản quyền KAFORECAST v6.0 | Tích hợp Overpass API Real-Data
            </div>
        </div>
    </div>
</div>

<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js" crossorigin=""></script>
<script src="https://unpkg.com/leaflet.heat/dist/leaflet-heat.js"></script>

<script>
    // =========================================================================
    // KAFORECAST JAVASCRIPT ENGINE
    // =========================================================================

    // Cờ kiểm tra xem hệ thống đã xuất dự báo (submit) hay chưa
    const hasPrediction = {% if prediction is not none %}true{% else %}false{% endif %};

    // Lấy tọa độ từ các Input ẩn (Đã được giữ lại qua Jinja nếu có submit)
    const initialLat = parseFloat(document.getElementById('lat_input').value) || 10.7769;
    const initialLng = parseFloat(document.getElementById('lng_input').value) || 106.7009;

    const map = L.map('map').setView([initialLat, initialLng], 15);

    // SỬA MÀU BẢN ĐỒ: Đổi từ dark_all sang light_all để hợp với giao diện sáng
    L.tileLayer('https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png', {
        attribution: '&copy; OpenStreetMap | KAFORECAST', maxZoom: 19
    }).addTo(map);

    let shopMarker = L.marker([initialLat, initialLng], { draggable: true, title: "Vị trí Cửa Hàng" }).addTo(map);
    let heatLayer = null;

    // Hàm cập nhật Input ẩn mỗi khi có sự thay đổi tọa độ
    function updateHiddenInputs(lat, lng) {
        document.getElementById('lat_input').value = lat;
        document.getElementById('lng_input').value = lng;
    }

    async function fetchRealGeoData(lat, lng) {
        const radius = 1000;
        const query = `
            [out:json][timeout:15];
            (
              node["amenity"~"school|university|college|kindergarten"](around:${radius},${lat},${lng});
              node["office"](around:${radius},${lat},${lng});
            );
            out body;
        `;
        const url = `https://overpass-api.de/api/interpreter?data=${encodeURIComponent(query)}`;

        try {
            const response = await fetch(url);
            if (!response.ok) throw new Error("Overpass API Error");
            const data = await response.json();

            let schools = 0; let offices = 0;
            let heatmap_points = [[lat, lng, 1.0]];

            if(data.elements && data.elements.length > 0) {
                data.elements.forEach(el => {
                    if(el.tags.amenity) { schools++; heatmap_points.push([el.lat, el.lon, 0.8]); }
                    else if(el.tags.office) { offices++; heatmap_points.push([el.lat, el.lon, 0.6]); }
                });
            }

            let type_id = 0; let context_name = "Khu Dân Cư Bình Thường";
            if (schools >= 3 && schools > offices * 0.5) {
                type_id = 1; context_name = "Khu vực Tập Trung Giáo Dục / Làng Đại Học";
            } else if (offices >= 4) {
                type_id = 2; context_name = "Khu Phố Văn Phòng / Kinh Doanh Nhộn Nhịp";
            }

            return {
                status: "success",
                context: { type_id: type_id, name: context_name, details: `<span class="text-success"><i class="fa-solid fa-database"></i> <b>KAFORECAST Live Data:</b></span> Quét thành công trong bán kính 1km phát hiện <b class='text-warning'>${schools} cơ sở giáo dục</b> và <b class='text-warning'>${offices} tòa nhà văn phòng</b> có thực trên vệ tinh.` },
                heatmap: heatmap_points
            };
        } catch (err) {
            return {
                status: "fallback",
                context: { type_id: 0, name: "Không Nhận Diện Được / Lỗi API", details: `<span class="text-danger"><i class="fa-solid fa-triangle-exclamation"></i> Lỗi máy chủ Vệ tinh.</span> Đã đặt mặc định.` },
                heatmap: [[lat, lng, 1.0]]
            };
        }
    }

    function updateLocationAnalysis(lat, lng) {
        document.getElementById('map-loading').style.display = 'flex';
        fetchRealGeoData(lat, lng).then(geoData => {
            document.getElementById('map-loading').style.display = 'none';

            const infoBox = document.getElementById('spatialInfoBox');
            infoBox.style.display = 'block';
            document.getElementById('spatialTitle').innerHTML = `<i class="fa-solid fa-location-crosshairs text-info"></i> Định Danh: <b>${geoData.context.name}</b>`;
            document.getElementById('spatialDetails').innerHTML = geoData.context.details;

            // Tự động điền f3
            const locationSelect = document.getElementById('f3');
            locationSelect.value = geoData.context.type_id;

            // Xóa heatmap cũ nếu có
            if(heatLayer) { map.removeLayer(heatLayer); }

            // ĐIỀU KIỆN MỚI: Chỉ render Heatmap NẾU đã có kết quả dự báo
            if (hasPrediction) {
                heatLayer = L.heatLayer(geoData.heatmap, {
                    radius: 35, blur: 25, maxZoom: 16,
                    gradient: { 0.2: 'blue', 0.4: 'cyan', 0.6: 'lime', 0.8: 'yellow', 1.0: 'red' }
                }).addTo(map);
            }

            shopMarker.bindPopup(`<div style="text-align:center;"><b>KAFORECAST POINT</b><br>${geoData.context.name}</div>`).openPopup();
        });
    }

    // Sự kiện kéo ghim
    shopMarker.on('dragend', function(e) {
        const pos = shopMarker.getLatLng();
        map.panTo(pos);
        updateHiddenInputs(pos.lat, pos.lng);
        updateLocationAnalysis(pos.lat, pos.lng);
    });

    // Sự kiện click bản đồ
    map.on('click', function(e) {
        shopMarker.setLatLng(e.latlng);
        map.panTo(e.latlng);
        updateHiddenInputs(e.latlng.lat, e.latlng.lng);
        updateLocationAnalysis(e.latlng.lat, e.latlng.lng);
    });

    // Tìm kiếm Nominatim
    document.getElementById('btnSearchLocation').addEventListener('click', function() {
        const query = document.getElementById('addressSearch').value;
        if(!query) return;

        document.getElementById('map-loading').style.display = 'flex';
        document.getElementById('loadingText').innerText = "Đang tìm kiếm tọa độ thật...";

        fetch(`https://nominatim.openstreetmap.org/search?format=json&q=${encodeURIComponent(query)}&limit=1`)
            .then(res => res.json())
            .then(data => {
                if(data && data.length > 0) {
                    const lat = parseFloat(data[0].lat);
                    const lon = parseFloat(data[0].lon);
                    map.setView([lat, lon], 16);
                    shopMarker.setLatLng([lat, lon]);
                    updateHiddenInputs(lat, lon);
                    updateLocationAnalysis(lat, lon);
                } else {
                    document.getElementById('map-loading').style.display = 'none';
                    alert("KAFORECAST không thể định vị!");
                }
            }).catch(err => {
                document.getElementById('map-loading').style.display = 'none';
                alert("Lỗi kết nối máy chủ bản đồ Nominatim.");
            });
    });

    // Chạy mặc định dựa trên tọa độ ban đầu (Tự động lấy từ hidden input)
    setTimeout(() => {
        updateLocationAnalysis(initialLat, initialLng);
    }, 800);

    // =========================================================================
    // CHART JS VÀ GIAO DIỆN NÚT BẤM
    // =========================================================================
    {% if prediction is not none %}
    document.addEventListener("DOMContentLoaded", function() {
        const ctx = document.getElementById('impactChart').getContext('2d');
        const chartData = {{ chart_data | safe }};

        // Đổi màu Label trong chart để hợp với nền sáng
        Chart.defaults.color = '#0f172a';
        Chart.defaults.font.family = "'Be Vietnam Pro', sans-serif";

        new Chart(ctx, {
            type: 'polarArea',
            data: {
                labels: ['Giá Bán', 'Lượng Khách', 'Chất Vị Trí', 'Chỗ Ngồi', 'Giao Hàng (App)', 'Nhân Sự', 'Khách Ngồi Lâu', 'Mặt Bằng (m2)', 'Cạnh Tranh'],
                datasets: [{
                    data: chartData,
                    backgroundColor: [
                        'rgba(255, 99, 132, 0.75)', 'rgba(78, 205, 196, 0.75)', 'rgba(255, 230, 109, 0.75)',
                        'rgba(26, 83, 92, 0.75)', 'rgba(247, 255, 247, 0.75)', 'rgba(255, 159, 28, 0.75)',
                        'rgba(46, 196, 182, 0.75)', 'rgba(0, 255, 204, 0.75)', 'rgba(231, 29, 54, 0.75)'
                    ],
                    borderColor: 'rgba(255,255,255,0.5)', borderWidth: 1.5, hoverBorderColor: '#ffffff'
                }]
            },
            options: {
                responsive: true, maintainAspectRatio: false,
                plugins: {
                    legend: { position: 'right', labels: { color: '#0f172a', font: { size: 12, weight: '600' } } },
                    tooltip: { callbacks: { label: function(context) { return ' ' + context.label + ': ' + context.raw + '% Ảnh Hưởng'; } } }
                },
                scales: { r: { ticks: { display: false }, grid: { color: 'rgba(0, 0, 0, 0.1)', lineWidth: 1 } } }
            }
        });

        window.scrollTo({ top: document.getElementById('resultSection').offsetTop - 30, behavior: 'smooth' });
    });
    {% endif %}

    document.getElementById('aiForm').addEventListener('submit', function() {
        const btn = document.getElementById('submitBtn');
        btn.innerHTML = '<i class="fa-solid fa-spinner fa-spin me-2"></i> KAFORECAST ĐANG TRÍCH XUẤT...';
        btn.style.opacity = '0.8'; btn.style.pointerEvents = 'none'; btn.style.boxShadow = 'none';
    });
</script>

</body>
</html>
"""

@app.route('/', methods=['GET'])
def index():
    return render_template_string(HTML_DASHBOARD, prediction=None, chart_data="[]")

@app.route('/process', methods=['POST'])
def process():
    try:
        inputs = [
            float(request.form.get('f1') or 0.0), float(request.form.get('f2') or 0.0),
            float(request.form.get('f3') or 0.0), float(request.form.get('f4') or 0.0),
            float(request.form.get('f5') or 0.0), float(request.form.get('f6') or 0.0),
            float(request.form.get('f7') or 0.0), float(request.form.get('f8') or 0.0),
            float(request.form.get('f9') or 0.0)
        ]

        revenue, percentage_list = kaforecast_engine.run_inference(inputs)
        chart_data_json = json.dumps(percentage_list)

        return render_template_string(HTML_DASHBOARD, prediction=revenue, chart_data=chart_data_json)

    except Exception as e:
        error_msg = f"Lỗi lõi KAFORECAST Engine: {str(e)}"
        print(f"[ERROR] {error_msg}")
        return render_template_string(HTML_DASHBOARD, prediction=None, error=error_msg)

def start_kaforecast_server():
    log = logging.getLogger('werkzeug')
    log.setLevel(logging.ERROR)
    app.run(port=5000, debug=False, use_reloader=False)

if __name__ == "__main__":
    server_thread = threading.Thread(target=start_kaforecast_server, daemon=True)
    server_thread.start()

    try:
        print("\n" + "★"*65)
        print(" KHỞI ĐỘNG HỆ THỐNG KAFORECAST REAL-TIME ".center(65, " "))
        print("★"*65)

        req = urllib.request.Request('https://ipv4.icanhazip.com', headers={'User-Agent': 'Mozilla/5.0'})
        endpoint_ip = urllib.request.urlopen(req, timeout=5).read().decode('utf8').strip()

        print(f"👉 BƯỚC 1: Hãy click vào link LocalTunnel được sinh ra ở ô Code bên dưới.")
        print(f"🔑 BƯỚC 2: Copy MẬT KHẨU IP này và dán vào trang web đó: [ {endpoint_ip} ]")
        print("★"*65 + "\n")

    except Exception as e:
        print(f"⚠️ Lỗi xác định IP: {e}")
!npx localtunnel --port 5000

[KAFORECAST] Đang tiến hành giả lập 1200 mẫu dữ liệu không gian kinh doanh...
[KAFORECAST] Bắt đầu quá trình khớp dữ liệu (Fitting Model)...
--- [KAFORECAST] Huấn luyện hoàn tất trong 0.11s ---
--- [KAFORECAST] Hệ số giao điểm (Intercept): -10106698.807461971 ---

★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
             KHỞI ĐỘNG HỆ THỐNG KAFORECAST REAL-TIME             
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
 * Serving Flask app '__main__'
 * Debug mode: off
👉 BƯỚC 1: Hãy click vào link LocalTunnel được sinh ra ở ô Code bên dưới.
🔑 BƯỚC 2: Copy MẬT KHẨU IP này và dán vào trang web đó: [ 104.199.131.225 ]
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋Need to install the following packages:
localtunnel@2.0.2
Ok to proceed? (y) Y

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏your url is: https://rich-baboons-brush.loca.lt
^C
